# ⏱️ Time-Aware Validation Analysis for Traffic Volume Prediction
### Evaluating Out-of-Time Model Generalization & Temporal Leakage

---

## 1. Objective & Motivation

In time-series and temporal forecasting problems, standard random train/test splitting can introduce **optimistic evaluation metrics**. When data is randomly shuffled, observations from adjacent hours or days (e.g., 2 PM and 3 PM on the same date) are randomly partitioned into both training and testing sets. This causes **temporal information leakage**, as the model is evaluated on test points that are temporally surrounded by training data.

### Core Objectives:
1. **Chronological Splitting**: Sort the dataset strictly by `date_time` and perform an out-of-time 85/15 train/test split (training on earlier historical data, testing on later unseen data).
2. **Production Feature Preservation**: Maintain the exact production feature set: `['hour', 'day', 'month', 'weekday', 'is_rush']`.
3. **Model Evaluation**: Train Linear Regression, Decision Tree, and Random Forest ($N=200$, depth=10) on historical data and evaluate on future test data.
4. **Comparative Analysis**: Compare chronological validation metrics against random 85/15 split metrics to quantify performance shifts under realistic time-aware evaluation.
5. **Methodological Discussion**: Analyze out-of-time generalization, temporal leakage risks, and real-world deployment considerations.

---

## 2. Environment Setup & Library Imports

In [3]:
import os
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor

# Configure plotting parameters
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['font.size'] = 11

warnings.filterwarnings('ignore')
print("✅ Libraries loaded successfully.")

✅ Libraries loaded successfully.


## 3. Data Preprocessing & Chronological Sorting

In [5]:
# Load raw dataset
df = pd.read_csv("datafile.csv")

# 1. Drop duplicate rows matching baseline pipeline
df_clean = df.drop_duplicates().copy()

# 2. Parse date_time
df_clean['date_time'] = pd.to_datetime(df_clean['date_time'], dayfirst=True)

# 3. Extract exact temporal features
df_clean['hour'] = df_clean['date_time'].dt.hour
df_clean['day'] = df_clean['date_time'].dt.day
df_clean['month'] = df_clean['date_time'].dt.month
df_clean['weekday'] = df_clean['date_time'].dt.weekday
df_clean['is_rush'] = df_clean['hour'].apply(lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0)

# 4. Sort strictly chronologically by date_time
df_sorted = df_clean.sort_values(by='date_time').reset_index(drop=True)

print("=== DATASET PREPROCESSING & SORTING VERIFICATION ===")
print(f"Total Clean Observations: {len(df_sorted)}")
print(f"Earliest Timestamp       : {df_sorted['date_time'].min()}")
print(f"Latest Timestamp         : {df_sorted['date_time'].max()}")
df_sorted[['date_time', 'hour', 'day', 'month', 'weekday', 'is_rush', 'traffic_volume']].head()

=== DATASET PREPROCESSING & SORTING VERIFICATION ===
Total Clean Observations: 48187
Earliest Timestamp       : 2012-10-02 09:00:00
Latest Timestamp         : 2018-09-30 23:00:00


## 4. Chronological Train/Test Partitioning (Out-of-Time Split)

In [7]:
# Production feature set
features = ['hour', 'day', 'month', 'weekday', 'is_rush']
X = df_sorted[features]
y = df_sorted['traffic_volume']

# Chronological split at 85% index (earlier observations = train, later = test)
split_index = int(len(df_sorted) * 0.85)

x_train_chrono = X.iloc[:split_index]
y_train_chrono = y.iloc[:split_index]
x_test_chrono = X.iloc[split_index:]
y_test_chrono = y.iloc[split_index:]

train_timestamps = df_sorted['date_time'].iloc[:split_index]
test_timestamps = df_sorted['date_time'].iloc[split_index:]

print("=== CHRONOLOGICAL SPLIT SUMMARY ===")
print(f"Train Set Observations : {len(x_train_chrono)} ({len(x_train_chrono)/len(df_sorted)*100:.2f}%)")
print(f"Test Set Observations  : {len(x_test_chrono)} ({len(x_test_chrono)/len(df_sorted)*100:.2f}%)")
print(f"Training Window Start  : {train_timestamps.min()}")
print(f"Training Window End    : {train_timestamps.max()}")
print(f"Testing Window Start   : {test_timestamps.min()}")
print(f"Testing Window End     : {test_timestamps.max()}")

=== CHRONOLOGICAL SPLIT SUMMARY ===
Train Set Observations : 40958 (85.00%)
Test Set Observations  : 7229 (15.00%)
Training Window Start  : 2012-10-02 09:00:00
Training Window End    : 2018-01-25 19:00:00
Testing Window Start   : 2018-01-25 19:00:00
Testing Window End     : 2018-09-30 23:00:00


## 5. Model Training & Chronological Evaluation

In [9]:
# Initialize models matching production configurations
lr_chrono = LinearRegression()
dt_chrono = DecisionTreeRegressor(random_state=42)
rf_chrono = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)

# Fit models on chronological training data
lr_chrono.fit(x_train_chrono, y_train_chrono)
dt_chrono.fit(x_train_chrono, y_train_chrono)
rf_chrono.fit(x_train_chrono, y_train_chrono)

chrono_models = {
    "Linear Regression": lr_chrono,
    "Decision Tree": dt_chrono,
    "Random Forest": rf_chrono
}

chrono_eval_results = []

for name, model in chrono_models.items():
    preds = model.predict(x_test_chrono)
    mae = mean_absolute_error(y_test_chrono, preds)
    mse = mean_squared_error(y_test_chrono, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_chrono, preds)
    
    chrono_eval_results.append({
        "Model": name,
        "MAE": round(mae, 4),
        "MSE": round(mse, 2),
        "RMSE": round(rmse, 4),
        "R2 Score": round(r2, 6)
    })

chrono_results_df = pd.DataFrame(chrono_eval_results)

print("=== CHRONOLOGICAL VALIDATION METRICS TABLE ===")
print(chrono_results_df.to_string(index=False))

=== CHRONOLOGICAL VALIDATION METRICS TABLE ===
            Model       MAE        MSE      RMSE  R2 Score
Linear Regression 1544.0189 3008837.69 1734.6002  0.235589
    Decision Tree  323.8007  332417.93  576.5570  0.915547
    Random Forest  264.1283  236940.51  486.7654  0.939804


## 6. Comparative Analysis: Random Split vs. Chronological Split

In [11]:
# Compute Random 85/15 Split metrics for direct side-by-side comparison
x_train_rand, x_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.15, random_state=42
)

rf_rand = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_rand.fit(x_train_rand, y_train_rand)
rf_rand_preds = rf_rand.predict(x_test_rand)

rand_mae = mean_absolute_error(y_test_rand, rf_rand_preds)
rand_mse = mean_squared_error(y_test_rand, rf_rand_preds)
rand_rmse = np.sqrt(rand_mse)
rand_r2 = r2_score(y_test_rand, rf_rand_preds)

chrono_rf_preds = rf_chrono.predict(x_test_chrono)
chrono_mae = mean_absolute_error(y_test_chrono, chrono_rf_preds)
chrono_mse = mean_squared_error(y_test_chrono, chrono_rf_preds)
chrono_rmse = np.sqrt(chrono_mse)
chrono_r2 = r2_score(y_test_chrono, chrono_rf_preds)

comp_df = pd.DataFrame({
    "Validation Method": ["Random Split (85/15)", "Chronological Split (85/15)"],
    "MAE": [round(rand_mae, 4), round(chrono_mae, 4)],
    "MSE": [round(rand_mse, 2), round(chrono_mse, 2)],
    "RMSE": [round(rand_rmse, 4), round(chrono_rmse, 4)],
    "R2 Score": [round(rand_r2, 6), round(chrono_r2, 6)]
})

print("=== RANDOM SPLIT VS CHRONOLOGICAL SPLIT (RANDOM FOREST) ===")
print(comp_df.to_string(index=False))

# Visualization of Metric Comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# MAE Comparison
sns.barplot(x="Validation Method", y="MAE", data=comp_df, ax=axes[0], palette="Blues_d")
axes[0].set_title("MAE: Random vs Chronological Split")
axes[0].set_ylabel("Mean Absolute Error")

# RMSE Comparison
sns.barplot(x="Validation Method", y="RMSE", data=comp_df, ax=axes[1], palette="Oranges_d")
axes[1].set_title("RMSE: Random vs Chronological Split")
axes[1].set_ylabel("Root Mean Squared Error")

# R2 Comparison
sns.barplot(x="Validation Method", y="R2 Score", data=comp_df, ax=axes[2], palette="Greens_d")
axes[2].set_title("R² Score: Random vs Chronological Split")
axes[2].set_ylabel("R² Score")

plt.tight_layout()
plt.show()

=== RANDOM SPLIT VS CHRONOLOGICAL SPLIT (RANDOM FOREST) ===
          Validation Method      MAE       MSE     RMSE  R2 Score
       Random Split (85/15) 257.8632 188481.31 434.1443  0.952749
Chronological Split (85/15) 264.1283 236940.51 486.7654  0.939804


## 7. Out-of-Time Forecasting Visualization

In [13]:
# Sample 7-day out-of-time test window for visual inspection (approx 168 hours)
sample_window = 168
test_sample_dates = test_timestamps.iloc[:sample_window]
test_sample_actual = y_test_chrono.iloc[:sample_window]
test_sample_preds = chrono_rf_preds[:sample_window]

plt.figure(figsize=(15, 6))
plt.plot(test_sample_dates, test_sample_actual, label="Actual Traffic Volume", color="black", alpha=0.8, linewidth=1.5)
plt.plot(test_sample_dates, test_sample_preds, label="Predicted Traffic Volume (Random Forest)", color="crimson", linestyle="--", linewidth=1.5)

plt.title("Out-of-Time Forecast vs. Actual Traffic Volume (First 7 Days of Test Set: Jan 2018)")
plt.xlabel("Date & Time")
plt.ylabel("Traffic Volume (vehicles/hour)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Detailed Methodological Discussion

### 1. Why Random Splitting Can Be Optimistic for Temporal Data
In time-series datasets, consecutive hourly observations exhibit **strong temporal autocorrelation**. When a random split is applied:
- Neighboring data points (e.g., 8 AM and 9 AM on October 15) are distributed across both training and test sets.
- The model evaluates test observations whose immediate temporal context was present during training, introducing **information leakage**.
- As a result, random split metrics ($R^2 = 0.952749, 	ext{RMSE} = 434.14$) tend to be slightly optimistic.

### 2. Why Chronological Validation is Necessary
- Chronological validation trains strictly on past data (`2012-10-02` to `2018-01-25`) and evaluates strictly on future unseen data (`2018-01-25` to `2018-09-30`).
- This accurately mimics **real-world production forecasting**, where future data is strictly unavailable during training.

### 3. Model Generalization Assessment
- Under chronological validation, the Random Forest model achieves **$R^2 = 0.939804$** ($	ext{RMSE} = 486.77$).
- Although $R^2$ decreases slightly by $1.29\%$ relative to random splitting, the model retains strong predictive accuracy across a 8-month unseen future window. This confirms that cyclical temporal features (`hour`, `weekday`, `is_rush`, `month`, `day`) provide robust generalized signals.

### 4. Real-World Deployment Disclaimer
- While a high chronological $R^2$ demonstrates temporal stability of historical patterns, **it does not guarantee production readiness on its own**.
- Real-world deployment requires continuous drift monitoring, handling structural shifts (e.g., road construction, pandemic travel shifts), and establishing real-time data ingestion pipelines.

## 9. Executive Summary & Takeaways

1. **Partitioning**: Trained on 40,958 historical records (`2012-10-02` to `2018-01-25`) and tested on 7,229 future records (`2018-01-25` to `2018-09-30`).
2. **Performance Shift**:
   - Random Split RF: $R^2 = 0.952749, 	ext{RMSE} = 434.14$
   - Chronological Split RF: $R^2 = 0.939804, 	ext{RMSE} = 486.77$
3. **Conclusion**: Random Forest demonstrates strong out-of-time generalization ($R^2 = 93.98\%$), confirming that temporal feature engineering captures stable recurrent traffic behavior over multi-year horizons.